# Monitor entity extraction results
Read-only. Per extraction run: the passage text, the latest successful Jev judgement for that passage (probability per bundle, ≥ threshold → selected), and the extraction response. Samples are drawn by status and by response size. Re-run the sampling cells (change `SEED`) for new samples.

In [1]:
from pathlib import Path
import json, random, textwrap
from sqlalchemy import text
from potency_kg.db import create_engine_from_env

nb_dir = Path.cwd().resolve()
root = nb_dir.parent if nb_dir.name == "notebooks" else nb_dir
scope = json.loads((root / "configs/casgevy-potency.json").read_text())["scope"]
PARAMS = {"tenant": scope["tenant_id"], "project": scope["project_id"]}
engine = create_engine_from_env()
SEED, N, TEXT_CHARS = 1, 3, 1500   # samples per bucket, passage chars shown
RESP_CHARS = 2500
print("scope:", PARAMS)

scope: {'tenant': 'public-fda', 'project': 'casgevy-potency'}


In [2]:
# One row per extraction run + passage + latest succeeded Jev response for the passage.
SQL = """
SELECT r.extraction_run_id, r.status, r.error_detail, r.attempts, r.model, r.selected_bundles, r.response,
       r.created_at, p.source_id, p.passage_id, p.text, p.locator,
       j.probabilities, j.criteria, j.questions,
       coalesce(jsonb_array_length(r.response->'entities'),0)      AS n_ent,
       coalesce(jsonb_array_length(r.response->'assertions'),0)    AS n_asr,
       coalesce(jsonb_array_length(r.response->'relationships'),0) AS n_rel,
       coalesce(jsonb_array_length(r.response->'source_refs'),0)   AS n_ref,
       coalesce(length(r.response::text),0)                        AS resp_len,
       (SELECT string_agg(DISTINCT v.code, ',') FROM validation_failure v
         WHERE v.tenant_id=r.tenant_id AND v.project_id=r.project_id AND v.stage='extraction'
           AND v.immutable_run_id=r.extraction_run_id)             AS failure_code
FROM llm_extraction_run r
JOIN source_passage p USING (tenant_id, project_id, source_id, source_version, passage_id)
LEFT JOIN LATERAL (SELECT * FROM jev_response j
                    WHERE j.tenant_id=r.tenant_id AND j.project_id=r.project_id AND j.source_id=r.source_id
                      AND j.source_version=r.source_version AND j.passage_id=r.passage_id AND j.status='succeeded'
                    ORDER BY j.created_at DESC LIMIT 1) j ON true
WHERE r.tenant_id=:tenant AND r.project_id=:project
"""
with engine.connect() as c:
    runs = [dict(x) for x in c.execute(text(SQL), PARAMS).mappings()]
for r in runs:
    r["empty"] = r["status"] == "succeeded" and r["n_ent"] + r["n_asr"] + r["n_rel"] + r["n_ref"] == 0
    r["bucket"] = "failed" if r["status"] == "failed" else ("succeeded-empty" if r["empty"] else "succeeded-nonempty")
print("runs:", len(runs))

runs: 746


## Overview

In [3]:
from collections import Counter
def table(counter, title):
    print(f"== {title} ==")
    for k, n in sorted(counter.items(), key=lambda kv: -kv[1]):
        print(f"  {str(k):40s} {n:5d}  {n/len(runs):6.1%}")
table(Counter(r["bucket"] for r in runs), "bucket")
table(Counter((r["failure_code"] or "-") for r in runs if r["status"]=="failed"), "failure_code (failed runs)")
table(Counter(r["attempts"] for r in runs), "attempts")
table(Counter(r["model"] for r in runs), "model")

# Does Jev selection explain empties/failures? bucket x number of selected bundles
print("== bucket x #bundles selected for extraction ==")
for b in ("failed","succeeded-empty","succeeded-nonempty"):
    print(" ", b, sorted(Counter(len(r["selected_bundles"]) for r in runs if r["bucket"]==b).items()))

# Per-bundle: how often is a bundle selected, and what does the run yield?
print("== per bundle: runs / failed / empty / non-empty ==")
bundles = sorted({b for r in runs for b in r["selected_bundles"]})
for b in bundles:
    rs = [r for r in runs if b in r["selected_bundles"]]
    c = Counter(r["bucket"] for r in rs)
    print(f"  {b:22s} {len(rs):5d}  failed={c['failed']:4d} empty={c['succeeded-empty']:4d} non-empty={c['succeeded-nonempty']:4d}")

# Response size distribution of succeeded runs
sizes = sorted(r["resp_len"] for r in runs if r["status"]=="succeeded")
if sizes:
    q = lambda p: sizes[min(len(sizes)-1, int(p*len(sizes)))]
    print("== response length (chars, succeeded) ==", {"min":sizes[0],"p25":q(.25),"median":q(.5),"p75":q(.75),"p90":q(.9),"max":sizes[-1]})

== bucket ==
  failed                                     611   81.9%
  succeeded-nonempty                          71    9.5%
  succeeded-empty                             64    8.6%
== failure_code (failed runs) ==
  validation_failed                          611   81.9%
== attempts ==
  1                                          746  100.0%
== model ==
  claude-sonnet-5-5                          746  100.0%
== bucket x #bundles selected for extraction ==
  failed [(1, 5), (2, 57), (3, 135), (4, 174), (5, 157), (6, 77), (7, 6)]
  succeeded-empty [(1, 25), (2, 26), (3, 13)]
  succeeded-nonempty [(1, 3), (2, 14), (3, 21), (4, 17), (5, 16)]
== per bundle: runs / failed / empty / non-empty ==
  alias_identity           717  failed= 594 empty=  54 non-empty=  69
  assay_evidence           382  failed= 341 empty=  12 non-empty=  29
  potency_strategy          57  failed=  54 empty=   1 non-empty=   2
  product_context          554  failed= 495 empty=  11 non-empty=  48
  redaction_uncerta

In [4]:
# Mean Jev probability per bundle by bucket: are failures/empties concentrated where Jev was unsure?
print(f"{'bundle':22s}" + "".join(f"{b:>22s}" for b in ("failed","succeeded-empty","succeeded-nonempty")))
for b in sorted({k for r in runs if r["probabilities"] for k in r["probabilities"]}):
    row = ""
    for bk in ("failed","succeeded-empty","succeeded-nonempty"):
        v = [r["probabilities"][b] for r in runs if r["bucket"]==bk and r["probabilities"] and b in r["probabilities"]]
        row += f"{(sum(v)/len(v) if v else float('nan')):22.2f}"
    print(f"{b:22s}{row}")

bundle                                failed       succeeded-empty    succeeded-nonempty
alias_identity                          0.92                  0.72                  0.90
assay_evidence                          0.55                  0.28                  0.43
potency_strategy                        0.13                  0.04                  0.08
product_context                         0.69                  0.25                  0.55
redaction_uncertainty                   0.69                  0.38                  0.73
regulatory_history                      0.46                  0.22                  0.32
review_conclusion                       0.45                  0.12                  0.25


## Show helper

In [5]:
def wrap(s, w=110, ind="    "):
    return "\n".join(textwrap.fill(l, w, initial_indent=ind, subsequent_indent=ind) if l.strip() else "" for l in str(s).splitlines())

def show(r, text_chars=None, resp_chars=None):
    text_chars = text_chars or TEXT_CHARS; resp_chars = resp_chars or RESP_CHARS
    print("=" * 120)
    print(f"{r['source_id']} / {r['passage_id']}   run={r['extraction_run_id']}   status={r['status']}   attempts={r['attempts']}   {r['created_at']:%Y-%m-%d %H:%M}")
    print(f"extraction asked for bundles: {r['selected_bundles']}")
    if r["status"] == "failed":
        print(f"failure: {r['error_detail']!r} code={r['failure_code']}")
    else:
        print(f"response: entities={r['n_ent']} assertions={r['n_asr']} relationships={r['n_rel']} source_refs={r['n_ref']} len={r['resp_len']}")
    print("-- Jev judgement (probability per bundle; * = extraction bundle) --")
    if r["probabilities"]:
        for b, p in sorted(r["probabilities"].items(), key=lambda kv: -kv[1]):
            star = "*" if b in r["selected_bundles"] else " "
            print(f"  {star} {p:5.2f} {b:22s} {(r['questions'] or {}).get(b,'')}")
    else:
        print("  (no successful Jev response)")
    t = r["text"]
    print(f"-- passage ({len(t)} chars) --\n{wrap(t[:text_chars])}{' …' if len(t) > text_chars else ''}")
    for d in (r["response"] or {}).get("dropped", []):
        print(f"  DROPPED {d['kind']} {d['local_id']}: {d['reason']}  quote={str((d['record'] or {}).get('supporting_quote'))[:80]!r}")
    print("-- extraction response --")
    if r["response"] is None:
        print("    null")
    else:
        s = json.dumps(r["response"], indent=2, ensure_ascii=False)
        print(wrap(s[:resp_chars], 140) + (" …" if len(s) > resp_chars else ""))

def sample(rows, n=N, seed=None, **kw):
    rng = random.Random(SEED if seed is None else seed)
    picked = rng.sample(rows, min(n, len(rows)))
    print(f"[{len(picked)} of {len(rows)}]")
    for r in picked: show(r, **kw)

## Samples by status

In [6]:
print("##### FAILED"); sample([r for r in runs if r["bucket"]=="failed"])

##### FAILED
[3 of 611]
casgevy-125785-clinical-pharmacology-bla-review-casgevy / casgevy-125785-clinical-pharmacology-bla-review-casgevy:page-0003-c01   run=a6ba3682-1850-449a-88df-c6d83372e5a8   status=failed   attempts=1   2026-10-13 05:56
extraction asked for bundles: ['product_context', 'alias_identity', 'assay_evidence', 'regulatory_history', 'review_conclusion', 'redaction_uncertainty']
failure: 'extraction validation failed' code=validation_failed
-- Jev judgement (probability per bundle; * = extraction bundle) --
  *  0.99 alias_identity         Does this passage contain a product name, abbreviation, or alias?
  *  0.94 assay_evidence         Does this passage describe assays or measured attributes?
  *  0.92 product_context        Does this passage identify product, application, indication, or applicability?
  *  0.91 review_conclusion      Does this passage state a reviewer or agency conclusion?
  *  0.90 redaction_uncertainty  Does this passage contain redaction or unresolv

In [7]:
print("##### SUCCEEDED, EMPTY RESPONSE"); sample([r for r in runs if r["bucket"]=="succeeded-empty"])

##### SUCCEEDED, EMPTY RESPONSE
[3 of 64]
casgevy-125785-pharmacovigilance-plan-review-memo-casgevy / casgevy-125785-pharmacovigilance-plan-review-memo-casgevy:page-0006-c02   run=965f556f-2779-44af-ab33-d141f2f9adb1   status=succeeded   attempts=1   2026-10-13 05:56
extraction asked for bundles: ['alias_identity']
response: entities=0 assertions=0 relationships=0 source_refs=0 len=74
-- Jev judgement (probability per bundle; * = extraction bundle) --
  *  0.87 alias_identity         Does this passage contain a product name, abbreviation, or alias?
     0.42 assay_evidence         Does this passage describe assays or measured attributes?
     0.08 product_context        Does this passage identify product, application, indication, or applicability?
     0.08 redaction_uncertainty  Does this passage contain redaction or unresolved uncertainty?
     0.02 review_conclusion      Does this passage state a reviewer or agency conclusion?
     0.02 regulatory_history     Does this passage recor

In [8]:
print("##### SUCCEEDED, NON-EMPTY"); sample([r for r in runs if r["bucket"]=="succeeded-nonempty"])

##### SUCCEEDED, NON-EMPTY
[3 of 71]
casgevy-125785-cber-dmpq-cmc-facility-bla-review-memo-casgevy / casgevy-125785-cber-dmpq-cmc-facility-bla-review-memo-casgevy:page-0067   run=a3e19b72-08e1-4244-9805-c524713ff4fd   status=succeeded   attempts=1   2026-10-13 05:56
extraction asked for bundles: ['product_context', 'alias_identity', 'regulatory_history', 'review_conclusion', 'redaction_uncertainty']
response: entities=1 assertions=0 relationships=0 source_refs=0 len=236
-- Jev judgement (probability per bundle; * = extraction bundle) --
  *  0.98 redaction_uncertainty  Does this passage contain redaction or unresolved uncertainty?
  *  0.96 alias_identity         Does this passage contain a product name, abbreviation, or alias?
  *  0.73 review_conclusion      Does this passage state a reviewer or agency conclusion?
  *  0.71 product_context        Does this passage identify product, application, indication, or applicability?
  *  0.52 regulatory_history     Does this passage record a 

## Samples by response length
Succeeded runs ranked by serialized response size: smallest non-empty, median, largest.

In [9]:
ok = sorted((r for r in runs if r["bucket"]=="succeeded-nonempty"), key=lambda r: r["resp_len"])
print("##### SHORTEST non-empty responses"); [show(r) for r in ok[:N]]

##### SHORTEST non-empty responses
casgevy-125785-cmc-review / casgevy-125785-cmc-review:page-0017   run=5fd61b99-013f-413f-8979-93d828bb5eaa   status=succeeded   attempts=1   2026-10-13 05:56
extraction asked for bundles: ['alias_identity', 'redaction_uncertainty']
response: entities=1 assertions=0 relationships=0 source_refs=0 len=233
-- Jev judgement (probability per bundle; * = extraction bundle) --
  *  0.94 redaction_uncertainty  Does this passage contain redaction or unresolved uncertainty?
  *  0.86 alias_identity         Does this passage contain a product name, abbreviation, or alias?
     0.38 product_context        Does this passage identify product, application, indication, or applicability?
     0.19 regulatory_history     Does this passage record a regulatory interaction or chronology?
     0.11 assay_evidence         Does this passage describe assays or measured attributes?
     0.08 potency_strategy       Does this passage discuss potency strategy?
     0.06 review_con

[None, None, None]

In [10]:
print("##### MEDIAN-length responses"); mid = len(ok)//2; [show(r) for r in ok[max(0, mid-N//2): mid-N//2+N]]

##### MEDIAN-length responses
sbra-casgevy-tdt-2024-01 / sbra-casgevy-tdt-2024-01:page-0009-c02   run=3a2a87bf-3187-479a-a5bf-7fa15bc5fd16   status=succeeded   attempts=1   2026-10-13 05:56
extraction asked for bundles: ['product_context', 'alias_identity']
response: entities=1 assertions=1 relationships=0 source_refs=1 len=1621
-- Jev judgement (probability per bundle; * = extraction bundle) --
  *  0.98 alias_identity         Does this passage contain a product name, abbreviation, or alias?
  *  0.80 product_context        Does this passage identify product, application, indication, or applicability?
     0.46 assay_evidence         Does this passage describe assays or measured attributes?
     0.34 redaction_uncertainty  Does this passage contain redaction or unresolved uncertainty?
     0.29 review_conclusion      Does this passage state a reviewer or agency conclusion?
     0.05 regulatory_history     Does this passage record a regulatory interaction or chronology?
     0.03 poten

[None, None, None]

In [11]:
print("##### LONGEST responses"); [show(r, resp_chars=4000) for r in ok[-N:][::-1]]

##### LONGEST responses
casgevy-125785-mid-cycle-summary / casgevy-125785-mid-cycle-summary:page-0006-c01   run=43e22bc5-943b-445b-879e-bb41735074e8   status=succeeded   attempts=1   2026-10-13 05:56
extraction asked for bundles: ['product_context', 'alias_identity', 'regulatory_history', 'review_conclusion', 'redaction_uncertainty']
response: entities=5 assertions=4 relationships=0 source_refs=5 len=5382
-- Jev judgement (probability per bundle; * = extraction bundle) --
  *  0.98 regulatory_history     Does this passage record a regulatory interaction or chronology?
  *  0.98 redaction_uncertainty  Does this passage contain redaction or unresolved uncertainty?
  *  0.97 alias_identity         Does this passage contain a product name, abbreviation, or alias?
  *  0.75 product_context        Does this passage identify product, application, indication, or applicability?
  *  0.47 review_conclusion      Does this passage state a reviewer or agency conclusion?
     0.27 assay_evidence    

[None, None, None]

## Hunt: Jev said "yes" but extraction came back empty or failed
Highest Jev probability among the bundles the extraction asked for — these are the clearest candidates for prompt / schema / grounding bugs.

In [12]:
def top_p(r): return max((r["probabilities"] or {}).get(b, 0) for b in r["selected_bundles"]) if r["probabilities"] else 0
suspects = sorted((r for r in runs if r["bucket"] in ("failed","succeeded-empty")), key=top_p, reverse=True)
print("##### EMPTY, most confident Jev"); [show(r) for r in [r for r in suspects if r["bucket"]=="succeeded-empty"][:N]]

##### EMPTY, most confident Jev
policy-pdufa7-goals-2022 / policy-pdufa7-goals-2022:page-0003   run=f973738f-5311-4c06-be6c-07a1d7888968   status=succeeded   attempts=1   2026-10-13 05:56
extraction asked for bundles: ['product_context', 'alias_identity']
response: entities=0 assertions=0 relationships=0 source_refs=0 len=74
-- Jev judgement (probability per bundle; * = extraction bundle) --
  *  0.97 alias_identity         Does this passage contain a product name, abbreviation, or alias?
  *  0.66 product_context        Does this passage identify product, application, indication, or applicability?
     0.38 regulatory_history     Does this passage record a regulatory interaction or chronology?
     0.22 review_conclusion      Does this passage state a reviewer or agency conclusion?
     0.22 redaction_uncertainty  Does this passage contain redaction or unresolved uncertainty?
     0.08 assay_evidence         Does this passage describe assays or measured attributes?
     0.02 potency_s

[None, None, None]

In [13]:
print("##### FAILED, most confident Jev"); [show(r) for r in [r for r in suspects if r["bucket"]=="failed"][:N]]

##### FAILED, most confident Jev
casgevy-125785-analytical-method-review / casgevy-125785-analytical-method-review:page-0001-c01   run=b222b0b0-15f0-4ef1-84f0-15d9b524b38a   status=failed   attempts=1   2026-10-13 05:56
extraction asked for bundles: ['product_context', 'alias_identity', 'regulatory_history', 'review_conclusion', 'redaction_uncertainty']
failure: 'extraction validation failed' code=validation_failed
-- Jev judgement (probability per bundle; * = extraction bundle) --
  *  0.99 alias_identity         Does this passage contain a product name, abbreviation, or alias?
  *  0.98 redaction_uncertainty  Does this passage contain redaction or unresolved uncertainty?
  *  0.97 review_conclusion      Does this passage state a reviewer or agency conclusion?
  *  0.88 product_context        Does this passage identify product, application, indication, or applicability?
  *  0.79 regulatory_history     Does this passage record a regulatory interaction or chronology?
     0.45 assay_ev

[None, None, None]

## Look up one passage
Set `PASSAGE_ID` to see every run (all attempts / statuses) for it.

In [14]:
PASSAGE_ID = ""   # e.g. a passage_id printed above
for r in sorted((r for r in runs if r["passage_id"] == PASSAGE_ID), key=lambda r: r["created_at"]): show(r, text_chars=4000)